# 05 Floors Gained — LightGBM

## Goal

Same setup as `04_win_rate_logistic_regression.ipynb` — a hand-picked list of "cards of interest" (fill in `CARDS_OF_INTEREST` below based on `03_exploratory_analysis.ipynb`'s screening output), restricted to occasions where the card was offered — but modeling `floors_gained` instead of `victory`, with LightGBM instead of a linear logistic regression, since `floors_gained` is continuous and its relationship with the confounders (especially `floor`) is unlikely to be linear.

Features: `was_picked`, `hp_ratio`, `floor`, `relic_count`, `ascension_level` — same set as 04.

**Deliberately excluded:** `floor_reached` is not a feature here, and this exclusion matters *more* than it did in 04. `floors_gained = floor_reached - floor` exactly, so if both `floor_reached` and `floor` were features, the model could reconstruct the target with zero error — that's not signal, it's leakage, and it would make `was_picked`'s importance meaningless. `floor` (the pick's own floor, known at decision time) stays in; `floor_reached` (a fact about how the run ended) does not.

**Reading the results:** LightGBM doesn't have a coefficient the way logistic regression does. `was_picked`'s effect here is estimated by predicting `floors_gained` twice for every row — once with `was_picked` forced to 1, once forced to 0, holding everything else fixed — and averaging the difference. This is a simple counterfactual/partial-dependence-style estimate, not a formal causal estimate, and there's no significance test on it yet (no bootstrap CI, unlike 04's p-values) — treat it as a first pass.

In [ ]:
import os
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up.
PROJECT_ROOT = Path.cwd().parent

os.environ["JAVA_HOME"] = str(PROJECT_ROOT / ".jdk17" / "jdk-17.0.20+8")
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = str(PROJECT_ROOT / ".venv" / "Scripts") + os.pathsep + r"C:\hadoop\bin" + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")

GOLD_CARD_CHOICE_EVENTS_PATH = str(PROJECT_ROOT / "raw_data" / "gold" / "card_choice_events")

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder.master("local[*]")
    .appName("floors-gained-lightgbm")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.shuffle.partitions", "100")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(GOLD_CARD_CHOICE_EVENTS_PATH)
print(f"Loaded {GOLD_CARD_CHOICE_EVENTS_PATH}")

## Cards of interest

Fill this in from `03_exploratory_analysis.ipynb`'s screening output — the cards that showed up with high `floors_gained_lift` and/or `win_rate_lift` (and enough sample size to trust the estimate). One row per (character, card) pair. Can be the same list as `04_win_rate_logistic_regression.ipynb`, or a different one if you want to focus this notebook on the floors_gained-specific top list.

In [ ]:
CARDS_OF_INTEREST = pd.DataFrame([
    # ("IRONCLAD", "Card Name"),
    # ("THE_SILENT", "Card Name"),
    # ("DEFECT", "Card Name"),
    # ("WATCHER", "Card Name"),
], columns=["character_chosen", "card_name"])

print("Cards of interest:", len(CARDS_OF_INTEREST))

In [ ]:
candidate_characters = CARDS_OF_INTEREST["character_chosen"].unique().tolist()
candidate_cards = CARDS_OF_INTEREST["card_name"].unique().tolist()

# Coarse filter in Spark (character/card lists, not exact pairs — colorless cards can appear
# for multiple characters), collect once, then narrow to exact (character, card) pairs and fit
# models in pandas. Avoids re-querying the full table once per candidate card. Note floor_reached
# is intentionally never selected here — see the leakage note above.
regression_pd = (
    df.filter(F.col("character_chosen").isin(candidate_characters) & F.col("card_name").isin(candidate_cards))
    .select("character_chosen", "card_name", "was_picked", "floors_gained", "floor", "current_hp", "max_hp", "relic_count", "ascension_level")
    .na.drop()
    .toPandas()
)
regression_pd = regression_pd.merge(CARDS_OF_INTEREST, on=["character_chosen", "card_name"], how="inner")
print("Collected rows for regression candidates:", len(regression_pd))

In [ ]:
regression_pd["was_picked"] = regression_pd["was_picked"].astype(int)
regression_pd = regression_pd[regression_pd["max_hp"] > 0].copy()
regression_pd["hp_ratio"] = regression_pd["current_hp"] / regression_pd["max_hp"]

FEATURE_COLS = ["was_picked", "hp_ratio", "floor", "relic_count", "ascension_level"]
MIN_REGRESSION_ROWS = 200

def fit_card_lightgbm(group):
    if len(group) < MIN_REGRESSION_ROWS:
        return pd.Series({"n": len(group), "error": "too few rows"})

    X = group[FEATURE_COLS]
    y = group["floors_gained"]
    try:
        model = lgb.LGBMRegressor(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.05,
            min_child_samples=20,
            verbosity=-1,
        )
        model.fit(X, y)
    except Exception as exc:
        return pd.Series({"n": len(group), "error": str(exc)})

    # Counterfactual effect estimate: predict floors_gained with was_picked forced to 1 vs 0
    # for every row, holding the other features fixed, and average the difference.
    X_picked = X.copy()
    X_picked["was_picked"] = 1
    X_not_picked = X.copy()
    X_not_picked["was_picked"] = 0
    effect = (model.predict(X_picked) - model.predict(X_not_picked)).mean()

    importance = dict(zip(FEATURE_COLS, model.feature_importances_))
    return pd.Series({
        "n": len(group),
        "error": None,
        "was_picked_effect": effect,
        "was_picked_importance": importance["was_picked"],
    })

results_pd = (
    regression_pd.groupby(["character_chosen", "card_name"])
    .apply(fit_card_lightgbm, include_groups=False)
    .reset_index()
)
print("Fitted:", (results_pd["error"].isna()).sum(), "of", len(results_pd))

### Results

`was_picked_effect` is the estimated change in `floors_gained` from picking the card, holding HP ratio, floor, relic count, and ascension fixed — the LightGBM analog of 04's odds ratio, but on the original `floors_gained` scale rather than log-odds. `was_picked_importance` is the raw LightGBM feature importance (split count) for `was_picked`, shown as a secondary signal of how much the model actually uses the feature versus, e.g., `floor` dominating. Sorted within each character by `was_picked_effect`; no significance filter applied yet (no p-value equivalent computed here — see the note above).

In [ ]:
fitted = results_pd[results_pd["error"].isna()].sort_values(
    ["character_chosen", "was_picked_effect"], ascending=[True, False]
)

cols = ["character_chosen", "card_name", "n", "was_picked_effect", "was_picked_importance"]
fitted[cols]

## Stop Spark

Run this when done exploring — otherwise the JVM stays alive holding memory until the kernel is restarted.

In [ ]:
spark.stop()